In [1]:
import os
import numpy as np
import pandas as pd
import utils
import string
ALPHABET_LIST = list(string.ascii_uppercase)

In [2]:
filepath = '/root/cernbox/backups/old_database/2023_07_17/pmt_202307171723.csv'
src_dir, filename = os.path.split(filepath)
data = pd.read_csv(filepath)
data

,serial_number,id_pmt_block,module,module_in_legacy_tilecal,pos_in_legacy_tilecal,pos_in_phase2_tilecal,beta,HV_nominal,QE,pmt_type,run_number,component_status,current_location,remark,created_at,updated_at,created_by,updated_by
0,110003,NaN,NaN,NaN,NaN,NaN,6.482,708.0,20.9,R11187\r,158.0,NaN,NaN,NaN,2021-03-31 11:24:24,2022-11-16 12:41:44,NaN,NaN
1,110005,NaN,NaN,NaN,NaN,NaN,6.581,687.0,20.6,R11187\r,158.0,NaN,NaN,NaN,2021-03-31 11:24:24,2021-03-31 09:24:24,NaN,NaN
2,110032,NaN,NaN,NaN,NaN,NaN,6.385,733.0,21.0,R11187\r,158.0,NaN,NaN,NaN,2021-03-31 11:24:24,2021-03-31 09:24:24,NaN,NaN
3,110041,NaN,NaN,NaN,NaN,NaN,6.688,641.0,20.5,R11187\r,158.0,NaN,NaN,NaN,2021-03-31 11:24:24,2021-03-31 09:24:24,NaN,NaN
4,110050,NaN,NaN,NaN,NaN,NaN,6.417,667.0,22.0,R11187\r,158.0,NaN,NaN,NaN,2021-03-31 11:24:24,2021-03-31 09:24:24,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10710,BB0017,NaN,NaN,EBA15,23.0,NaN,-1.000,-1.0,NaN,R7877,NaN,NaN,NaN,NaN,2021-03-31 11:24:15,2021-03-31 09:24:15,NaN,NaN
10711,BB0166,NaN,NaN,EBA51,14.0,NaN,-1.000,-1.0,NaN,R7877,NaN,NaN,NaN,NaN,2021-03-31 11:24:15,2021-03-31 09:24:15,NaN,NaN
10712,null01,NaN,NaN,NaN,NaN,NaN,6.776,637.0,23.5,R11187\r,164.0,NaN,NaN,NaN,2021-03-31 11:24:25,2021-03-31 09:24:25,NaN,NaN
10713,XX0018,NaN,NaN,NaN,0.0,NaN,-1.000,642.0,NaN,R7877,NaN,NaN,NaN,NaN,2021-03-31 11:24:15,2021-03-31 09:24:15,NaN,NaN


In [3]:
data['new_id'] = pd.NaT
data['new_type'] = 'pmt'
data['new_owner'] = 'CERN'
data['new_location'] = pd.NaT
data['new_status'] = pd.NaT
data['new_serial'] = pd.NaT
data['new_beta'] = np.nan
data['new_hv_nominal'] = np.nan
data['new_qe'] = np.nan
data['new_model'] = pd.NaT
data['new_comments'] = ''

## General Checks

In [4]:
data.shape

(10715, 29)

In [5]:
data.dtypes

serial_number                       object
id_pmt_block                       float64
module                             float64
module_in_legacy_tilecal            object
pos_in_legacy_tilecal              float64
pos_in_phase2_tilecal              float64
beta                               float64
HV_nominal                         float64
QE                                 float64
pmt_type                            object
run_number                         float64
component_status                    object
current_location                    object
remark                              object
created_at                          object
updated_at                          object
created_by                         float64
updated_by                         float64
new_id                      datetime64[ns]
new_type                            object
new_owner                           object
new_location                datetime64[ns]
new_status                  datetime64[ns]
new_serial 

In [6]:
null_count = data.isnull().sum()
null_count[null_count < len(data)]  #Selects columns that have non null values

serial_number                   0
module_in_legacy_tilecal     1044
pos_in_legacy_tilecal         448
beta                            0
HV_nominal                      0
QE                          10315
pmt_type                        0
run_number                  10315
component_status            10628
current_location            10389
remark                      10536
created_at                      0
updated_at                      0
new_type                        0
new_owner                       0
new_comments                    0
dtype: int64

## Serial number valid?

### Serials have multiple regex matches

We either have serial that match the expected regex, serial that are full numerical that were made numeric with the mapping of the first 2 letters to their index in the alphabet or there is a strange value called null01

In [7]:
correct_regex = '[A-Z]{2}[0-9]{4}'
correct_match = utils.match_regex('serial_number', correct_regex, data)
number_regex = '[0-9]{6}'
number_match = utils.match_regex('serial_number', number_regex, data)
data.loc[~(correct_match | number_match)]

,serial_number,id_pmt_block,module,module_in_legacy_tilecal,pos_in_legacy_tilecal,pos_in_phase2_tilecal,beta,HV_nominal,QE,pmt_type,...,new_type,new_owner,new_location,new_status,new_serial,new_beta,new_hv_nominal,new_qe,new_model,new_comments
10712,null01,NaN,NaN,NaN,NaN,NaN,6.776,637.0,23.5,R11187\r,...,pmt,CERN,NaT,NaT,NaT,NaN,NaN,NaN,NaT,


### Cleaning

In [8]:
data.loc[:, 'new_serial'] = data['serial_number']

In [9]:
def map_back_to_alphabet(serial_number: str):
    serial_list = list(serial_number)
    serial_list[0] = ALPHABET_LIST[int(serial_list[0])]
    serial_list[1] = ALPHABET_LIST[int(serial_list[1])]
    return ''.join(serial_list)

In [10]:
data.loc[number_match, 'new_serial'] = data.loc[number_match, 'serial_number'].apply(
    map_back_to_alphabet
)
data = data.loc[data['new_serial'] != 'null01']

#### Fixed? (Yes)

In [11]:
utils.is_unique('new_serial', data)

True

In [12]:
data.loc[~data['new_serial'].str.match(correct_regex), 'new_serial']

Series([], Name: new_serial, dtype: object)

## Beta is valid?

In [13]:
beta_is_positive = utils.greater_than('beta', 0, data)
data.loc[~beta_is_positive, 'beta'].value_counts(dropna=False)

beta
-1.000    1013
 0.000     258
-4.406       1
-4.817       1
Name: count, dtype: int64

In [14]:
data.loc[beta_is_positive, 'new_beta'] = data.loc[beta_is_positive, 'beta']

In [15]:
(data['new_beta'] <= 0).any()

False

## HV nominal is valid?

In [16]:
hv_is_positive = utils.greater_than('HV_nominal', 0, data)
data.loc[~hv_is_positive, 'HV_nominal'].value_counts(dropna=False)

HV_nominal
-1.0    13
 0.0     2
Name: count, dtype: int64

In [17]:
data.loc[hv_is_positive, 'new_hv_nominal'] = data.loc[hv_is_positive, 'HV_nominal']

In [18]:
(data['new_hv_nominal'] <= 0).any()

False

## QE is valid?

All non positive QE values are NaN

In [19]:
qe_is_positive = utils.greater_than('QE', 0, data)
data.loc[~qe_is_positive, 'QE'].value_counts(dropna=False)

QE
NaN    10315
Name: count, dtype: int64

In [20]:
data.loc[qe_is_positive, 'new_qe'] = data.loc[qe_is_positive, 'QE']

In [21]:
(data['new_qe'] <= 0).any()

False

## Type is valid?
There are some pmt types that end in `\r` for some reason, does this means something or just multiple typos? I assumed is just a type and replaced it

In [22]:
data['pmt_type'].value_counts()

pmt_type
R7877       10315
R11187\r      395
R11187          4
Name: count, dtype: int64

In [23]:
data.loc[:, 'new_model'] = data['pmt_type'].str.replace('\r', '')
data['new_model'].value_counts()

new_model
R7877     10315
R11187      399
Name: count, dtype: int64

## Location is Valid?
There are multple places that map to the same place, those values need to be standardized.
There are some unknown values:

In [24]:
data.loc[:, 'new_location'] = data['current_location'].str.replace(' ', '_').str.lower()
data['new_location'].value_counts(dropna=False)

new_location
NaN                    10388
cern_175                 130
ux15                      40
bratislava                29
cern_testbeam             29
pisa_q&a_grid             23
cern_171                  21
pisa                      11
lhcb                      11
bratislava_?              10
should_be_at_cern         10
pisa_long_term_test        4
pisa_hvad_test             2
cern_174                   1
on_detector                1
cluj                       1
cenr_171                   1
bin                        1
cern_171_                  1
Name: count, dtype: int64

In [25]:
location_map = {
    'cern_171': 'B171',
    'cenr_171': 'B171',
    'cern_171_': 'B171',
    'cern_174': 'B175',
    'cern_175': 'B175',
    'pisa': 'Universita & INFN, Pisa',
    'ux15': 'UX15',
    'on_detector': 'ATLAS',
    'cern_testbeam': 'Testbeam',
    'lhcb': 'LHCb',
    'cluj': 'TIM, Cluj Napoca',
    'bratislava': 'Comenius University Bratislava',
    'bin': 'Unknown'
}

def uniformize_location_edge_cases(row: pd.Series) -> str:
    location = row['new_location']
    if pd.isna(location):
        row['new_location'] = 'Unknown'
    elif location.startswith('pisa'):
        splitted = location.split('_')
        row['new_comments'] += '_'.join(splitted[1:])
        row['new_location'] = location_map['pisa']
    elif location == 'should_be_at_cern':
        row['new_location'] = 'Unknown'
        if row['new_comments']:
            row['new_comments'] += ';Should be in CERN'
        else:
            row['new_comments'] += 'Should be in CERN'
    elif location == 'bratislava_?':
        row['new_location'] = 'Unknown'
        if row['new_comments']:
            row['new_comments'] += ';Should be in Bratislava'
        else:
            row['new_comments'] += 'Should be in Bratislava'
    else:
        row['new_location'] = location
    
    return row


def uniformize_locations(row: pd.Series) -> pd.Series:
    try:
        row['new_location'] = location_map[row['new_location']]
        return row
    except KeyError:
        return uniformize_location_edge_cases(row)

data = data.apply(uniformize_locations, axis=1)
data

,serial_number,id_pmt_block,module,module_in_legacy_tilecal,pos_in_legacy_tilecal,pos_in_phase2_tilecal,beta,HV_nominal,QE,pmt_type,...,new_type,new_owner,new_location,new_status,new_serial,new_beta,new_hv_nominal,new_qe,new_model,new_comments
0,110003,NaN,NaN,NaN,NaN,NaN,6.482,708.0,20.9,R11187\r,...,pmt,CERN,Unknown,NaT,BB0003,6.482,708.0,20.9,R11187,
1,110005,NaN,NaN,NaN,NaN,NaN,6.581,687.0,20.6,R11187\r,...,pmt,CERN,Unknown,NaT,BB0005,6.581,687.0,20.6,R11187,
2,110032,NaN,NaN,NaN,NaN,NaN,6.385,733.0,21.0,R11187\r,...,pmt,CERN,Unknown,NaT,BB0032,6.385,733.0,21.0,R11187,
3,110041,NaN,NaN,NaN,NaN,NaN,6.688,641.0,20.5,R11187\r,...,pmt,CERN,Unknown,NaT,BB0041,6.688,641.0,20.5,R11187,
4,110050,NaN,NaN,NaN,NaN,NaN,6.417,667.0,22.0,R11187\r,...,pmt,CERN,Unknown,NaT,BB0050,6.417,667.0,22.0,R11187,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10709,BB0015,NaN,NaN,EBA15,14.0,NaN,-1.000,-1.0,NaN,R7877,...,pmt,CERN,Unknown,NaT,BB0015,NaN,NaN,NaN,R7877,
10710,BB0017,NaN,NaN,EBA15,23.0,NaN,-1.000,-1.0,NaN,R7877,...,pmt,CERN,Unknown,NaT,BB0017,NaN,NaN,NaN,R7877,
10711,BB0166,NaN,NaN,EBA51,14.0,NaN,-1.000,-1.0,NaN,R7877,...,pmt,CERN,Unknown,NaT,BB0166,NaN,NaN,NaN,R7877,
10713,XX0018,NaN,NaN,NaN,0.0,NaN,-1.000,642.0,NaN,R7877,...,pmt,CERN,Unknown,NaT,XX0018,NaN,642.0,NaN,R7877,


In [26]:
data['new_location'].value_counts()

new_location
Unknown                           10409
B175                                131
Universita & INFN, Pisa              40
UX15                                 40
Comenius University Bratislava       29
Testbeam                             29
B171                                 23
LHCb                                 11
ATLAS                                 1
TIM, Cluj Napoca                      1
Name: count, dtype: int64

## Status is valid?

In [27]:
data['new_status'] = data['component_status'].str.lower().str.replace(' ', '_')
data['new_status'].value_counts(dropna=False)

new_status
NaN                  10627
removed                 57
being_tested            20
broken                   4
high_dark_current        2
bad_linearity            2
hv_problems              1
removed,_broken          1
Name: count, dtype: int64

In [28]:
status_map = {
    'removed': 'Stored',
    'being_tested': 'Testing',
    'broken': 'Broken',
    'removed,_broken': 'Broken'
}

def uniformize_status_edge_cases(row: pd.Series) -> str:
    status = row['new_status']
    problem_type = ['high_dark_current', 'bad_linearity', 'hv_problems']
    if pd.isna(status):
        row['new_status'] = 'Unknown'
    elif status in problem_type:
        row['new_status'] = 'Failure'
        row['new_comments'] += ';' + status
    else:
        row['new_status'] = status
    
    return row


def uniformize_status(row: pd.Series) -> pd.Series:
    try:
        row['new_status'] = status_map[row['new_status']]
        return row
    except KeyError:
        return uniformize_status_edge_cases(row)

data = data.apply(uniformize_status, axis=1)
data

,serial_number,id_pmt_block,module,module_in_legacy_tilecal,pos_in_legacy_tilecal,pos_in_phase2_tilecal,beta,HV_nominal,QE,pmt_type,...,new_type,new_owner,new_location,new_status,new_serial,new_beta,new_hv_nominal,new_qe,new_model,new_comments
0,110003,NaN,NaN,NaN,NaN,NaN,6.482,708.0,20.9,R11187\r,...,pmt,CERN,Unknown,Unknown,BB0003,6.482,708.0,20.9,R11187,
1,110005,NaN,NaN,NaN,NaN,NaN,6.581,687.0,20.6,R11187\r,...,pmt,CERN,Unknown,Unknown,BB0005,6.581,687.0,20.6,R11187,
2,110032,NaN,NaN,NaN,NaN,NaN,6.385,733.0,21.0,R11187\r,...,pmt,CERN,Unknown,Unknown,BB0032,6.385,733.0,21.0,R11187,
3,110041,NaN,NaN,NaN,NaN,NaN,6.688,641.0,20.5,R11187\r,...,pmt,CERN,Unknown,Unknown,BB0041,6.688,641.0,20.5,R11187,
4,110050,NaN,NaN,NaN,NaN,NaN,6.417,667.0,22.0,R11187\r,...,pmt,CERN,Unknown,Unknown,BB0050,6.417,667.0,22.0,R11187,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10709,BB0015,NaN,NaN,EBA15,14.0,NaN,-1.000,-1.0,NaN,R7877,...,pmt,CERN,Unknown,Unknown,BB0015,NaN,NaN,NaN,R7877,
10710,BB0017,NaN,NaN,EBA15,23.0,NaN,-1.000,-1.0,NaN,R7877,...,pmt,CERN,Unknown,Unknown,BB0017,NaN,NaN,NaN,R7877,
10711,BB0166,NaN,NaN,EBA51,14.0,NaN,-1.000,-1.0,NaN,R7877,...,pmt,CERN,Unknown,Unknown,BB0166,NaN,NaN,NaN,R7877,
10713,XX0018,NaN,NaN,NaN,0.0,NaN,-1.000,642.0,NaN,R7877,...,pmt,CERN,Unknown,Unknown,XX0018,NaN,642.0,NaN,R7877,


In [29]:
data['new_status'].value_counts()

new_status
Unknown    10627
Stored        57
Testing       20
Broken         5
Failure        5
Name: count, dtype: int64

## What are the remarks?

In [30]:
data.loc[~data['remark'].isnull(), 'remark'].value_counts()

remark
Has been in Pisa                                              27
On loan to Yuri Guz                                            9
October 2020                                                   5
https://pcata007.cern.ch/elog/Maintenance/4217                 4
https://pcata007.cern.ch/elog/Maintenance/4216                 4
                                                              ..
LBC28 PMT34 https://pcata007.cern.ch/elog/Maintenance/4269     1
LBC47 PMT48 https://pcata007.cern.ch/elog/Maintenance/4260     1
PMT block 700203\nEB MD2 pos 8                                 1
LBC63 PMT48 https://pcata007.cern.ch/elog/Maintenance/4284     1
EBA49 PMT11 https://pcata007.cern.ch/elog/Maintenance/4461     1
Name: count, Length: 113, dtype: int64

In [31]:
def append_remarks_to_comments(row: pd.Series):
    remark = row['remark']
    comment = row['new_comments']
    if pd.isna(remark):
        return row
    elif len(comment) == 0:
        row['new_comments'] = remark
    else:
        row['new_comments'] += ';' + remark

    return row

In [32]:
data = data.apply(append_remarks_to_comments, axis=1)
data['new_comments'].value_counts()

new_comments
                                                              10507
q&a_grid                                                         17
Should be in Bratislava;Has been in Pisa                         10
Has been in Pisa                                                  9
On loan to Yuri Guz                                               9
                                                              ...  
PMT block 100972\nEB MD2 pos 5                                    1
PMT block 601368\nEB MD1 pos 9                                    1
PMT block 700779\nEB MD1 pos 12                                   1
EBC 59 pmt 13                                                     1
EBA01 PMT11 https://pcata007.cern.ch/elog/Maintenance/4481        1
Name: count, Length: 123, dtype: int64

## Cleaned Data

In [33]:
cleaned_data = data.loc[:, data.columns[data.columns.str.startswith('new')]]
new_column_names = {
    old: '_'.join(old.split('_')[1:])
    for old in cleaned_data.columns
}
cleaned_data.rename(new_column_names, axis=1, inplace=True)
cleaned_data.loc[:, 'id'] = '20LOCER0' + cleaned_data['serial']
cleaned_data

,id,type,owner,location,status,serial,beta,hv_nominal,qe,model,comments
0,20LOCER0BB0003,pmt,CERN,Unknown,Unknown,BB0003,6.482,708.0,20.9,R11187,
1,20LOCER0BB0005,pmt,CERN,Unknown,Unknown,BB0005,6.581,687.0,20.6,R11187,
2,20LOCER0BB0032,pmt,CERN,Unknown,Unknown,BB0032,6.385,733.0,21.0,R11187,
3,20LOCER0BB0041,pmt,CERN,Unknown,Unknown,BB0041,6.688,641.0,20.5,R11187,
4,20LOCER0BB0050,pmt,CERN,Unknown,Unknown,BB0050,6.417,667.0,22.0,R11187,
...,...,...,...,...,...,...,...,...,...,...,...
10709,20LOCER0BB0015,pmt,CERN,Unknown,Unknown,BB0015,NaN,NaN,NaN,R7877,
10710,20LOCER0BB0017,pmt,CERN,Unknown,Unknown,BB0017,NaN,NaN,NaN,R7877,
10711,20LOCER0BB0166,pmt,CERN,Unknown,Unknown,BB0166,NaN,NaN,NaN,R7877,
10713,20LOCER0XX0018,pmt,CERN,Unknown,Unknown,XX0018,NaN,642.0,NaN,R7877,


In [34]:
empty_comment = cleaned_data['comments'].str.len() == 0
cleaned_data.loc[empty_comment,'comments'] = pd.NaT
cleaned_data

,id,type,owner,location,status,serial,beta,hv_nominal,qe,model,comments
0,20LOCER0BB0003,pmt,CERN,Unknown,Unknown,BB0003,6.482,708.0,20.9,R11187,NaN
1,20LOCER0BB0005,pmt,CERN,Unknown,Unknown,BB0005,6.581,687.0,20.6,R11187,NaN
2,20LOCER0BB0032,pmt,CERN,Unknown,Unknown,BB0032,6.385,733.0,21.0,R11187,NaN
3,20LOCER0BB0041,pmt,CERN,Unknown,Unknown,BB0041,6.688,641.0,20.5,R11187,NaN
4,20LOCER0BB0050,pmt,CERN,Unknown,Unknown,BB0050,6.417,667.0,22.0,R11187,NaN
...,...,...,...,...,...,...,...,...,...,...,...
10709,20LOCER0BB0015,pmt,CERN,Unknown,Unknown,BB0015,NaN,NaN,NaN,R7877,NaN
10710,20LOCER0BB0017,pmt,CERN,Unknown,Unknown,BB0017,NaN,NaN,NaN,R7877,NaN
10711,20LOCER0BB0166,pmt,CERN,Unknown,Unknown,BB0166,NaN,NaN,NaN,R7877,NaN
10713,20LOCER0XX0018,pmt,CERN,Unknown,Unknown,XX0018,NaN,642.0,NaN,R7877,NaN


In [35]:
cleaned_data.isnull().sum()

id                0
type              0
owner             0
location          0
status            0
serial            0
beta           1273
hv_nominal       15
qe            10315
model             0
comments      10507
dtype: int64

In [36]:
cleaned_data['location'].value_counts()

location
Unknown                           10409
B175                                131
Universita & INFN, Pisa              40
UX15                                 40
Comenius University Bratislava       29
Testbeam                             29
B171                                 23
LHCb                                 11
ATLAS                                 1
TIM, Cluj Napoca                      1
Name: count, dtype: int64

In [37]:
now = pd.Timestamp.now()
cleaned_filename = f'{now.strftime("%Y_%m_%d_%H_%M_%S")}_cleaned_{filename}'
cleaned_filepath = os.path.join('output', cleaned_filename)
print(cleaned_filepath)
cleaned_data.to_csv(cleaned_filepath)

output/2023_09_19_15_14_02_cleaned_pmt_202307171723.csv
